In [ ]:
#Latex display work arround for VSCode
from latex_disp_VSCode import slatex

In [1]:
# Function to create a list of [x,y] variables names
# Input: n 
# Oupout: List of strings [x1, ... xn, y1, ..., yn]
def XYvar(n):
    X=[]
    Y=[]
    for i in range(n):
        X.append('x'+str(i))
        Y.append('y'+str(i))
    X.extend(Y)
    return X

def ZWvar(n):
    Z=[]
    W=[]
    for i in range(n):
        Z.append('z'+str(i))
        W.append('w'+str(i))
    Z.extend(W)
    return Z

def XYZWvar(n, m):
    X=[]
    X = XYvar(n)
    X.extend(ZWvar(m))
    return X

In [ ]:
XYZWvar(3, 1)

In [ ]:
#Code to find solutions of superfrieze-systems in two odd variables \sigma \theta for the Kronecker triangulation
#Inputs: 
# n is the number of different marked points (2n are the indices for x's and y's)
# m is the period of the superfrieze, it satisfies that n = k m for some k in Z
# A is a vector of the a_i coefficients of the m odd variables that are of the form ai \sigma + bi \theta
# B is a vector of the b_i coefficients of the m odd variables
# L is the vector of the l_i coefficients that appear in the expression 3+ l\sigma\theta)
# A, B and L all must be of size m.
#Optional arguments
# groebner, always set to False. 
# ord, choice of monomial order, always set to 'degrevlex'
# helpb, always set to False, change to True if you want to see the reduction steps of the Groebner basis algorithm. If you want to the the steps, write
# from sage.misc.verbose import set_verbose
# set_verbose(1)
# at the beginning of your cell

# IMPORTANT: The algorithm used is the original Buchberger, it was used to be able to print the reduction steps. 

# Main tool: Grobner basis sage package

#Output: 
# If groebner = True, get the equations associated to the Kronecker superfrieze system. 
# Optional, if helpb = True, get the non-trivial steps of the Groebner basis algorithm. 
# Otherwise, the function outputs whether the system has a solution, by checking if 1 is in a grobner basis of the associated system. 

from sage.rings.polynomial.toy_buchberger import *
from sage.misc.verbose import set_verbose
from io import StringIO
import sys
def Kronecker_sys(A, B, L, n, groebner = False, ord = 'degrevlex', helpb = False, rexp = False):
    if len(A) != len(B):
        print ('A and B must be the sale lenght')
    else:
        m = len(A)
        if n % m != 0:
            print('m must divide n')
        else: 
            P = PolynomialRing(QQ, 4*n, XYvar(2*n), order=ord)
            P.inject_variables(verbose = False)
            eqsX = []
            eqsY = []
            eqsXY = []
            eqs = []
            nn=2*n
            for i in range(n):
                #append(P('x'+str((i)%n)+'+x'+str((i+1)%n)-A[i%m]))
                eqsX.append(P('x'+str((2*i)%nn)+'+x'+str((2*i+1)%nn)+'+x'+str((2*i+2)%nn))-A[i%m])
                eqsY.append(P('y'+str((2*i)%nn)+'+y'+str((2*i+1)%nn)+'+y'+str((2*i+2)%nn))-B[i%m])
                eqsXY.append(P('x'+str((2*i+1)%nn)+'*(y'+str((2*i+2)%nn)+'-y'+str((2*i)%nn)+')+y'+str((2*i+1)%nn)+'*(x'+str((2*i)%nn)+'-x'+str((2*i+2)%nn)+')+x'+str((2*i)%nn)+'*y'+str((2*i+2)%nn)+'-y'+str((2*i)%nn)+'*x'+str((2*i+2)%nn))-L[i%m])
            eqs.extend(eqsX)
            eqs.extend(eqsY)
            eqs.extend(eqsXY)
            if groebner :
                if helpb : 
                    buf = StringIO()
                    set_verbose(1)
                    old_stdout = sys.stdout
                    sys.stdout = buf
                    B = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
                    sys.stdout = old_stdout
                    set_verbose(0)
                    for line in buf.getvalue().splitlines():
                        if '=> 0' not in line and 'G:' not in line and line.strip():  # keep only non-trivial
                            print(line)
                    return [eqs, B]
                else : 
                    B = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
                    return [eqs, B]
            else:
                B = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
                if rexp :
                    return not (1 in B)
                else : 
                    if 1 in B:
                        print('The system is not solvable')
                    else: 
                        print('The system has a solution')


In [ ]:
# Example of system with no solution
A=[2,2]
B=[3,3]
L=[0, 1]
result = Kronecker_sys(A, B, L, 4, helpb=True, groebner=True)
for i in range(len(result[1])):
    print(result[1][i])
result[0]

In [ ]:
#Example of the previous system with more marked points, now the system has solutions
A=[3,3]
B=[3,3]
L=[0,1]
result = Kronecker_sys(A, B, L, 4, groebner=True, helpb=True)


In [ ]:
#Example: The sum of the special elements of the grobner basis stay when a1 \neq a2 and b1 \neq b2
A=[1,1,1]
B=[1,1,1]
L=[1,2,3]
R = Kronecker_sys(A, B, L, 3, groebner=True, helpb= True)
for i in range(len(R[1])):
    print(R[1][i])

In [ ]:
#Example: The sum of the special elements of the grobner basis stay when a1 = a2 and b1 \neq b2
A=[0,0]
B=[0,1]
L=[1,1]
Kronecker_sys(A, B, L, 2)

In [ ]:
#Example: The sum of the special elements of the grobner basis stay when a1 \neq a2 and b1 = b2
A=[0,0]
B=[0,0]
L=[1,1]
result = Kronecker_sys(A, B, L, 4, groebner=True)
for i in range(len(result[1])):
    print(result[1][i])

In [ ]:
A=[2,2,2]
B=[1,1,1]
L=[0,0,0]
result = Kronecker_sys(A, B, L, 3, groebner=True, helpb= True)
for i in range(len(result[1])):
  print(result[1][i])

In [ ]:
from sage.misc.prandom import randrange
from sage.misc.randstate import current_randstate

progress = {'i': 0, 'last': None}   # defined in global scope
def test_m(k, q, p) :
    for i in range(k):
        progress['i'] = i
        m = randint(2,q)
        A = [choice(list(range(1, p))) for j in range(m)]
        B = [choice(list(range(1, p))) for j in range(m)]
        L = [choice(list(range(1, p))) for j in range(m)]
        progress['last'] = (m, A, B, L)
        if not Kronecker_sys(A, B, L, m, rexp=True) : 
            print('This system has no solution')
            print([m,A,B,L])
            print('_________') 
#test_m(1000, 4, 5)

### Example of a A(1,3) frieze

In [ ]:
# Quadratic system of equations for an annuli
# Wedge equations (for qiddity inside of a wedge)

# def wedge(A,B,L,n, j = 0):
#     """ System associated to a wedge configuration. 
#     n - number of triangles in the wedge
#     If using for "doubled" triangulations, set j = n+1
#     """
#     if len(A) != len(B):
#         print ('A and B must be the same lenght')
#     else:
#         m = len(L)
#         if n % m != 0:
#             print('m must divide n')
#         else: 
#             eqsX = []
#             eqsY = []
#             eqsXY = []
#             eqs = []
#             for i in range(n):
#                 eqsX.append(P('x'+str((i+j)%n)+'+x'+str((i+j+1)%n))-A[i%m])
#                 eqsY.append(P('y'+str((i+j)%n)+'+y'+str((i+j+1)%n))-B[i%m])
#                 eqsXY.append(P('x'+str((i+j)%n)+'*y'+str((i+j+1)%n)+'-x'+str((j+i+1)%n)+'*y'+str((j+i)%n))-L[i%m])
#             eqs.extend(eqsX)
#             eqs.extend(eqsY)
#             eqs.extend(eqsXY)
#             return eqs


In [2]:
#General code for triangulations with only wedges ans 3-vees
P = PolynomialRing(QQ, 40, XYZWvar(15, 5))
P.inject_variables(verbose = False)
def wedge_sys(A,B,L,n, m=1, N = []):
    """ System associated to a wedge triangulation.  
    n - number of marked points, and hence wedge triangules
    m - number of wedges, default to 1
    N - vector of marked points where wedges in range(n) are (in order), default N=[0]
    """
    if len(A) != len(B):
        print ('A and B must be the same lenght')
    else:
        k = len(L)
        if n % k != 0:
            print('len(L) must divide n')
        else: 
            P = PolynomialRing(QQ, 2*n + 2*m, XYZWvar(n,m))
            P.inject_variables(verbose = False)
            N = [0]+N
            s = len(N)
            Np = [(N[i+1]-N[i]) for i in range(s-1)] + [n-N[s-1]]
            eqsX = []
            eqsY = []
            eqsXY = []
            eqsZ = []
            eqsZW = []
            eqs = []
            #print(N, Np)
            for i in range(s):
                r = N[i]
                for j in range(Np[i]-1):
                    eqsX.append(P('x'+str((j+r)%n)+'+x'+str((j+r+1)%n))-A[(j+r)%k])
                    eqsY.append(P('y'+str((j+r)%n)+'+y'+str((j+r+1)%n))-B[(j+r)%k])
                    eqsXY.append(P('x'+str((j+r)%n)+'*y'+str((j+r+1)%n)+'-x'+str((j+r+1)%n)+'*y'+str((j+r)%n))-L[(j+r)%k])
                eqsZ.append(P('x'+str((Np[i]-1+r)%n)+'+x'+str((Np[i]+r)%n)+'+z'+str(i%m))-A[(Np[i]-1+r)%k])
                eqsZ.append(P('y'+str((Np[i]-1+r)%n)+'+y'+str((Np[i]+r)%n)+'+w'+str(i%m))-B[(Np[i]-1+r)%k])
                eqsZW.append(P('x'+str((Np[i]-1+r)%n)+'*w'+str(i%s)+'-y'+str((Np[i]-1+r)%n)+'*z'+str(i%s)) + P('x'+str((Np[i]-1+r)%n)+'*y'+str((Np[i]+r)%n)+'-x'+str((Np[i]+r)%n)+'*y'+str((Np[i]-1+r)%n)) + P('-x'+str((Np[i]+r)%n)+'*w'+str(i%s)+'+y'+str((Np[i]+r)%n)+'*z'+str(i%s)) - L[(Np[i]-1+r)%k])
            eqs.extend(eqsX)
            eqs.extend(eqsY)
            eqs.extend(eqsXY)
            eqs.extend(eqsZ)
                #('x'+str((j+r)%n)+'*y'+str((j+r+1)%n)+'-x'+str((j+r+1)%n)+'*y'+str((j+r)%n))-L[(j+r)%k]
            eqs.extend(eqsZW)
            return eqs

In [ ]:
A = [choice(list(range(1, 10))) for j in range(3)]
B = [choice(list(range(1, 10))) for j in range(3)]
L = [choice(list(range(1, 10))) for j in range(3)]
eqs = wedge_sys(A,B,L,3, 1)
A, B, L, eqs

In [ ]:
#set_verbose(1)
GB = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
#for i in range(len(GB)):
#    print(GB[i])

In [ ]:
eqsd = wedge_sys(A+A,B+B,L+L,6, 2,[3])
eqsd

In [ ]:
GBd = (P.ideal(eqsd)).groebner_basis(algorithm = 'toy:buchberger')
#for i in range(len(GBd)):
#    print(GBd[i])

In [ ]:
eqst = wedge_sys(A+A+A,B+B+B,L+L+L,9, 3,[3,6])
eqst

In [ ]:
GBt = (P.ideal(eqst)).groebner_basis(algorithm = 'toy:buchberger')
for i in range(len(GBt)):
    print(GBt[i])

In [ ]:

eqsq = wedge_sys(A+A+A+A,B+B+B+B,L+L+L+L,12, 4,[3,6,9])
GBq = (P.ideal(eqsq)).groebner_basis(algorithm = 'toy:buchberger')
#for i in range(len(GBq)):
#    print(GBq[i])

In [ ]:
I = P.ideal(eqsq)
I.ring().gens()


Stepts in the Groebner basis argument:
(-x2*y1 + x1*y2 - 2, y1 + y2 - 5) => 6
G: {x0 + x1 - 1, 2*x2 + z0 - 10, y0 + y2 + w0 - 6, 6, x1 + x2 - 6, -x2*y1 + x1*y2 - 2, y1 + y2 - 5, y0 + y1 - 7, -2*y2 - w0 + 4, x0 + x2 + z0 - 5, z0 + 5/2*w0 + 14, x2*y0 - x0*y2 + y0*z0 - y2*z0 - x0*w0 + x2*w0 - 6, 37/4*w0 + 44, -x1*y0 + x0*y1 - 1}


(-x2*y1 + x1*y2 - 2, x2 + x3 + z0 - 5) => 6
G: {6, -x2*y1 + x1*y2 - 2, -x4*y3 + x3*y4 - 1, x4 + x5 - 6, -w0 + w1, -x1*y0 + x0*y1 - 1, 7*z1 - w1 + 10, y5*z1 - x5*w1 - 6*x5 + 5*y5 - 6*z1 + 5*w1 + 6, x3 + x4 - 1, x1 + x2 - 6, 7*x5 - y5 + 7*z1 - w1 - 28, 37/7*w1 + 176/7, y2 + y3 + w0 - 6, y4 + y5 - 5, -x3*y2 + x2*y3 - y2*z0 + y3*z0 + x2*w0 - x3*w0 - 6, x5*y0 - x0*y5 + y0*z1 - y5*z1 - x0*w1 + x5*w1 - 6, y3 + y4 - 7, -19/7*y5 - 4*w1 - 50/7, x0 + x1 - 1, x0 + x5 + z1 - 5, y1 + y2 - 5, x2 + x3 + z0 - 5, y0 + y5 + w1 - 6, -z0 + z1, y0 + y1 - 7, -x5*y4 + x4*y5 - 2}

(x2 + x3 + z0 - 5, -x2*y1 + x1*y2 - 2) => -111/14
G: {7*x11 - y11 + 7*z3 - w3 - 28, -x2*y1 + x1*y2 - 2, y7 + y8 - 5, -112/19*w3 - 18686/703, -x4*y3 + x3*y4 - 1, w0 - w1 + w2 - w3, x4 + x5 - 6, -x1*y0 + x0*y1 - 1, x6 + x7 - 1, -x8*y7 + x7*y8 - 2, -x7*y6 + x6*y7 - 1, x3 + x4 - 1, x10 + x11 - 6, x1 + x2 - 6, y6 + y7 - 7, y2 + y3 + w0 - 6, y4 + y5 - 5, -x3*y2 + x2*y3 - y2*z0 + y3*z0 + x2*w0 - x3*w0 - 6, -y11*z3 + 1/7*y11*w3 - z3*w3 + 1/7*w3^2 - 29/7*y11 - 1/7*w3 + 18, y3 + y4 - 7, y9 + y10 - 7, y8 + y9 + w2 - 6, y0 + y11 + w3 - 6, y11*z2 - x11*w2 - 6*x11 + 5*y11 - 4*z2 + 10*w2 + 34, x0 + x1 - 1, z0 - z1 + z2 - z3, -35/37*z2^2 + 35/37*z2*z3 + 47/37*z2*w2 - 42/37*z3*w2 - 6/37*w2^2 - 5/37*z2*w3 + 6/37*w2*w3 + 59/37*z2 - 77/37*z3 - 56/37*w2 + 11/37*w3 - 138/37, y1 + y2 - 5, 7*z1 - 7*z2 + 7*z3 - w1 + w2 - w3 + 10, x5 + x6 + z1 - 5, y10 + y11 - 5, x2 + x3 + z0 - 5, -37/7*y11 + 5*z2 - 5*z3 - 6*w2 + 5/7*w3 - 8, -111/14, y5 + y6 + w1 - 6, x8 + x9 + z2 - 5, -x6*y5 + x5*y6 - y5*z1 + y6*z1 + x5*w1 - x6*w1 - 6, -x9*y8 + x8*y9 - y8*z2 + y9*z2 + x8*w2 - x9*w2 - 6, 95/37*z2 - 114/37*w2 - 9/7*w3 - 4776/259, x9 + x10 - 1, -x11*y10 + x10*y11 - 2, 37/7*w1 - 37/7*w2 - 333/133*w3 - 786/133, x11*y0 - x0*y11 + y0*z3 - y11*z3 - x0*w3 + x11*w3 - 6, y0 + y1 - 7, x7 + x8 - 6, -7*z3 + w3 - 10, -x5*y4 + x4*y5 - 2, -x10*y9 + x9*y10 - 1, x0 + x11 + z3 - 5}



In [ ]:
# Try the smaller system 
A = [choice(list(range(1, 10))) for j in range(2)]
B = [choice(list(range(1, 10))) for j in range(2)]
L = [choice(list(range(1, 10))) for j in range(2)]
eqs = wedge_sys(A,B,L,2, 1)
A, B, L, eqs

([2, 6],
 [3, 8],
 [6, 8],
 [x0 + x1 - 2,
  y0 + y1 - 3,
  -x1*y0 + x0*y1 - 6,
  x0 + x1 + z0 - 6,
  y0 + y1 + w0 - 8,
  x1*y0 - x0*y1 + y0*z0 - y1*z0 - x0*w0 + x1*w0 - 8])

In [7]:
GB = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
GB

[x1*y0 - x0*y1 + y0*z0 - y1*z0 - x0*w0 + x1*w0 - 8,
 x0 + x1 + z0 - 6,
 x0 + x1 - 2,
 x1 - 10/13*y1 - 6/13,
 y0 + y1 + w0 - 8,
 y0 + y1 - 3,
 y1 + 24,
 z0 - 4,
 w0 - 5,
 x1*y0 - x0*y1 + 6]